# Colab #2：`pallier2025_full`（`sub-01..sub-10`，SimpleB2T `run 1–6 / 7 / 8–9`）

**推荐双阶段省算力流程**：
1. **阶段 A（CPU High-RAM 实例）**：逐被试流式物化 50 Hz 信号 + `t5-large` 词向量，写完即删原始 FIF/H5（峰值盘占 <10 GB），打包单文件 `features_pallier2025_full.tar`（约 3.6 GB）存入 `MyDrive/BrainDecoding_v2/archives/`。
2. **阶段 B（切换至 GPU L4 / A100 实例）**：重新挂载云盘与环境（无需重下 `t5-large`），秒级解包 `.tar` 到 `/content/local_cache`，串行训练并评估第 1 批 **`R1 → R3 → R4`**（`seed=0`）。
3. **阶段 C（确认 `R4 vs R1` 增幅清晰后运行）**：补齐第 2 批 **`R2`（端到端不冻结）** 与 **联合合成平滑脉冲流 `S1–S4`**。

> 如做临时调参试错，请将 `BRAINDECODING_RESULTS_DIR` 改为 `/content/drive/MyDrive/BrainDecoding_v2/scratch_runs`，勿污染正式 `runs/`。

## 阶段 A（CPU 实例）：物化并打包 `.tar` 存云盘
> 若 `MyDrive/BrainDecoding_v2/archives/features_pallier2025_full.tar` 已存在，直接跳到 **阶段 B**。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
git clone https://github.com/wdongdong0708-jpg/BrainDecoding2.git /content/BrainDecoding || (cd /content/BrainDecoding && git pull)
cd /content/BrainDecoding
python3 scripts/colab_setup.py --venv-dir /content/proj-venv --local-cache-dir /content/local_cache

# 仅物化阶段需要拉一次 t5-large（计算出的词向量会一并打进 .tar）
/content/proj-venv/bin/python -c "from transformers import AutoTokenizer, AutoModelForTextEncoding; AutoTokenizer.from_pretrained('t5-large'); AutoModelForTextEncoding.from_pretrained('t5-large')"

In [ ]:
%%bash
set -e
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/runs
ARCHIVE_DIR=/content/drive/MyDrive/BrainDecoding_v2/archives
PY=/content/proj-venv/bin/python

# 逐被试流式物化 sub-01..sub-10（写完 50 Hz .npy 即删临时 FIF/H5，防 Colab 爆盘）
$PY -u -m braindecoding cache \
  --dataset pallier2025_full \
  --subjects sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10 \
  --split-policy simpleb2t \
  --pnpl-root /content/tmp_pnpl_raw/pallier2025_full \
  --include-test --cleanup-raw

# 打包单一 .tar 持久化到 Google Drive（约 3.6 GB）
$PY scripts/cache_archive.py pack \
  --modality pallier2025_full \
  --cache-dir /content/local_cache \
  --archive-dir "$ARCHIVE_DIR"

ls -lh "$ARCHIVE_DIR/features_pallier2025_full.tar"

## 阶段 B（切换 GPU L4 / A100 实例后执行）：解包并跑第 1 批 `R1 → R3 → R4`
> 切换运行时后 `/content` 会清空，因此先重新执行下面 3 个轻量准备 Cell（无需再下载 `t5-large`）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
nvidia-smi
git clone https://github.com/wdongdong0708-jpg/BrainDecoding2.git /content/BrainDecoding || (cd /content/BrainDecoding && git pull)
cd /content/BrainDecoding
python3 scripts/colab_setup.py --venv-dir /content/proj-venv --local-cache-dir /content/local_cache

In [ ]:
%%bash
set -e
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
ARCHIVE_DIR=/content/drive/MyDrive/BrainDecoding_v2/archives
PY=/content/proj-venv/bin/python

$PY scripts/cache_archive.py unpack \
  --modality pallier2025_full \
  --cache-dir /content/local_cache \
  --archive-dir "$ARCHIVE_DIR" \
  --skip-existing

In [ ]:
%%bash
set -e
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/runs
PY=/content/proj-venv/bin/python

# R1: 词编码器（Word-Only）
$PY -u -m braindecoding train \
  --config configs/training/pallier2025_full_word_encoder.yaml \
  --exp-tag ch2_real_w10 --seed 0 --require-cache --resume auto

# R3: 词编码器冻结 + 整句联合编码（预提取冻结 CNN 特征提速 5-8x）
$PY -u -m braindecoding train \
  --config configs/training/pallier2025_full_context_transformer.yaml \
  --exp-tag ch2_real_frozen_w10 --word-exp-tag ch2_real_w10 \
  --preextract-frozen-cnn --seed 0 --require-cache --resume auto

# R4: 词编码器冻结 + 整句联合编码 + 式(1)-(2) 错位词间隔窗口去偏
$PY -u -m braindecoding train \
  --config configs/training/siamese_debiased/pallier2025_full_siamese_debiased_context.yaml \
  --exp-tag ch2_real_debias_w10 --word-exp-tag ch2_real_w10 \
  --seed 0 --require-cache --resume auto

In [ ]:
%%bash
set -e
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/runs
PY=/content/proj-venv/bin/python

# 第 1 批留出 test 集统一单次前向评估（eta_eval=0.0 + Pre-Roll 负对照）
$PY -u -m braindecoding eval-run \
  --config configs/training/pallier2025_full_word_encoder.yaml \
  --exp-tag ch2_real_w10 --seed 0 --split test --require-cache --no-plot

$PY -u -m braindecoding eval-run \
  --config configs/training/pallier2025_full_context_transformer.yaml \
  --exp-tag ch2_real_frozen_w10 --word-exp-tag ch2_real_w10 --seed 0 --split test \
  --control-config configs/controls/temporal_shift/temporal_shift_control.yaml \
  --shift-seconds -10,-20,-30,-40 --shift-mode preroll --require-cache --no-plot

$PY -u -m braindecoding eval-run \
  --config configs/training/siamese_debiased/pallier2025_full_siamese_debiased_context.yaml \
  --exp-tag ch2_real_debias_w10 --word-exp-tag ch2_real_w10 --seed 0 --split test \
  --control-config configs/controls/temporal_shift/temporal_shift_control.yaml \
  --shift-seconds -10,-20,-30,-40 --shift-mode preroll --require-cache --no-plot

## 阶段 C（可选，第 1 批确认趋势后执行）：第 2 批 `R2` + 联合合成信号 `S1–S4`

In [ ]:
%%bash
set -e
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/runs
PY=/content/proj-venv/bin/python

# R2: 随机初始化端到端不冻结 CNN + 16 层 Transformer
$PY -u -m braindecoding train \
  --config configs/training/pallier2025_full_context_transformer.yaml \
  --exp-tag ch2_real_unfrozen_w10 --from-scratch --seed 0 --require-cache --resume auto

$PY -u -m braindecoding eval-run \
  --config configs/training/pallier2025_full_context_transformer.yaml \
  --exp-tag ch2_real_unfrozen_w10 --seed 0 --split test --require-cache --no-plot

In [ ]:
%%bash
set -e
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/runs
PY=/content/proj-venv/bin/python

# S1 -> S2 -> S3 -> S4: 联合合成平滑脉冲流（零读取真实 MEG）
$PY -u -m braindecoding train \
  --config configs/training/pallier2025_full_word_encoder.yaml \
  --exp-tag ch2_synth_w10 --synthetic-timing shared --seed 0 --require-cache --resume auto

$PY -u -m braindecoding train \
  --config configs/training/pallier2025_full_context_transformer.yaml \
  --exp-tag ch2_synth_unfrozen_w10 --from-scratch --synthetic-timing shared --seed 0 --require-cache --resume auto

$PY -u -m braindecoding train \
  --config configs/training/pallier2025_full_context_transformer.yaml \
  --exp-tag ch2_synth_frozen_w10 --word-exp-tag ch2_synth_w10 --synthetic-timing shared --seed 0 --require-cache --resume auto

$PY -u -m braindecoding train \
  --config configs/training/siamese_debiased/pallier2025_full_siamese_debiased_context.yaml \
  --exp-tag ch2_synth_debias_w10 --word-exp-tag ch2_synth_w10 --synthetic-timing shared --seed 0 --require-cache --resume auto

# 留出 test 集评估（自动从检查点继承 synthetic_timing=shared）
$PY -u -m braindecoding eval-run --config configs/training/pallier2025_full_word_encoder.yaml --exp-tag ch2_synth_w10 --seed 0 --split test --require-cache --no-plot
$PY -u -m braindecoding eval-run --config configs/training/pallier2025_full_context_transformer.yaml --exp-tag ch2_synth_unfrozen_w10 --seed 0 --split test --require-cache --no-plot
$PY -u -m braindecoding eval-run --config configs/training/pallier2025_full_context_transformer.yaml --exp-tag ch2_synth_frozen_w10 --word-exp-tag ch2_synth_w10 --seed 0 --split test --require-cache --no-plot
$PY -u -m braindecoding eval-run --config configs/training/siamese_debiased/pallier2025_full_siamese_debiased_context.yaml --exp-tag ch2_synth_debias_w10 --word-exp-tag ch2_synth_w10 --seed 0 --split test --require-cache --no-plot